In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

import mlflow
import dagshub
import mlflow.sklearn



In [2]:
# MLflow
dagshub.init(
    repo_owner='IzaKakhniashvili',
    repo_name='ML-assignment1-HousePrices',
    mlflow=True
)


Accessing as IzaKakhniashvili

Initialized MLflow to track repo "IzaKakhniashvili/ML-assignment1-HousePrices"

Repository IzaKakhniashvili/ML-assignment1-HousePrices initialized!

In [3]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

## Ex3_skewed Features

### Data Cleaning

In [4]:
y = train["SalePrice"]
X = train.drop("SalePrice", axis=1)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
X_tr = X_train.copy()
X_v = X_val.copy()

In [6]:
num_cols = X_tr.select_dtypes(include=[np.number]).columns
cat_cols = X_tr.select_dtypes(include=['object']).columns

X_tr[num_cols] = X_tr[num_cols].fillna(X_tr[num_cols].median())
X_v[num_cols] = X_v[num_cols].fillna(X_tr[num_cols].median())
X_tr[cat_cols] = X_tr[cat_cols].fillna("None")
X_v[cat_cols] = X_v[cat_cols].fillna("None")

### Feature Engineering

In [7]:
for df in [X_tr, X_v]:
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df.fillna(0, inplace=True)

In [8]:
X_tr['is_train'] = 1
X_v['is_train'] = 0
combined = pd.concat([X_tr, X_v])
combined_encoded = pd.get_dummies(combined, columns=cat_cols)

X_tr_enc = combined_encoded[combined_encoded['is_train'] == 1].drop('is_train', axis=1)
X_v_enc = combined_encoded[combined_encoded['is_train'] == 0].drop('is_train', axis=1)

In [9]:
numeric_feats = X_tr_enc.dtypes[X_tr_enc.dtypes != "object"].index
skewed_feats = X_tr_enc[numeric_feats].apply(lambda x: x.skew()).sort_values(ascending=False)

# 0.75-ზე მეტი ასიმეტრია
high_skew = skewed_feats[abs(skewed_feats) > 0.75].index

In [10]:
for feat in high_skew:
    X_tr_enc[feat] = np.log1p(X_tr_enc[feat])
    X_v_enc[feat] = np.log1p(X_v_enc[feat])

In [11]:
y_train_log = np.log1p(y_train)

### Feature Selection

In [12]:
temp_df = pd.concat([X_tr_enc, y_train], axis=1)
corr_matrix = temp_df.corr()
threshold = 0.2
relevant_cols = corr_matrix['SalePrice'][abs(corr_matrix['SalePrice']) > threshold].index.tolist()
if 'SalePrice' in relevant_cols: relevant_cols.remove('SalePrice')

relevant_cols

['LotFrontage',
 'LotArea',
 'OverallQual',
 'YearBuilt',
 'YearRemodAdd',
 'MasVnrArea',
 'BsmtUnfSF',
 'TotalBsmtSF',
 '1stFlrSF',
 'GrLivArea',
 'BsmtFullBath',
 'FullBath',
 'HalfBath',
 'TotRmsAbvGrd',
 'Fireplaces',
 'GarageYrBlt',
 'GarageCars',
 'GarageArea',
 'WoodDeckSF',
 'OpenPorchSF',
 'TotalSF',
 'TotalBath',
 'HouseAge',
 'MSZoning_RL',
 'MSZoning_RM',
 'LotShape_Reg',
 'Neighborhood_NoRidge',
 'Neighborhood_NridgHt',
 'Neighborhood_StoneBr',
 'HouseStyle_2Story',
 'RoofStyle_Gable',
 'RoofStyle_Hip',
 'Exterior1st_VinylSd',
 'Exterior2nd_VinylSd',
 'MasVnrType_None',
 'MasVnrType_Stone',
 'ExterQual_Ex',
 'ExterQual_Gd',
 'ExterQual_TA',
 'Foundation_CBlock',
 'Foundation_PConc',
 'BsmtQual_Ex',
 'BsmtQual_Gd',
 'BsmtQual_TA',
 'BsmtExposure_Gd',
 'BsmtExposure_No',
 'BsmtFinType1_GLQ',
 'HeatingQC_Ex',
 'HeatingQC_TA',
 'CentralAir_N',
 'CentralAir_Y',
 'Electrical_SBrkr',
 'KitchenQual_Ex',
 'KitchenQual_Gd',
 'KitchenQual_TA',
 'FireplaceQu_Ex',
 'FireplaceQu_Gd',
 '

In [13]:
X_tr_final = X_tr_enc[relevant_cols]
X_v_final = X_v_enc[relevant_cols]

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_final)
X_v_scaled = scaler.transform(X_v_final)

In [15]:
model = LinearRegression()
model.fit(X_tr_scaled, y_train_log)

log_preds = model.predict(X_v_scaled)
final_preds = np.expm1(log_preds)

In [16]:
from sklearn.metrics import mean_squared_log_error

rmse = np.sqrt(mean_squared_log_error(y_val, final_preds))
rmse

np.float64(0.14752333199329176)

In [17]:
r2 = r2_score(y_val, final_preds)
r2

0.8993851686019833

In [18]:
mae = mean_absolute_error(y_val, final_preds)
mae

17359.27937095937

In [19]:
with mlflow.start_run(run_name="Exp_B"):
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("stage", "Experiment 3")
    mlflow.log_param("corr_threshold", threshold)
    mlflow.log_param("skew_threshold", 0.75)
    mlflow.log_param("final_features", len(relevant_cols))
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )

2026/04/15 21:11:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/15 21:11:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'house_price_model' already exists. Creating a new version of this model...
2026/04/15 21:11:37 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: house_price_model, version 22
Created version '22' of model 'house_price_model'.


🏃 View run Exp_B at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0/runs/607c4405717444f695288a2ad2966599
🧪 View experiment at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0


## Ex4_LASSO

### Data Cleaning

In [31]:
y = train["SalePrice"]
X = train.drop("SalePrice", axis=1)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

X_tr = X_train.copy()
X_v = X_val.copy()
num_cols = X_tr.select_dtypes(include=[np.number]).columns
cat_cols = X_tr.select_dtypes(include=['object']).columns

X_tr[num_cols] = X_tr[num_cols].fillna(X_tr[num_cols].median())
X_v[num_cols] = X_v[num_cols].fillna(X_tr[num_cols].median())
X_tr[cat_cols] = X_tr[cat_cols].fillna("None")
X_v[cat_cols] = X_v[cat_cols].fillna("None")

### Feature Engineering

In [32]:
for df in [X_tr, X_v]:
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df.fillna(0, inplace=True)

In [33]:
X_tr['is_train'] = 1
X_v['is_train'] = 0
combined = pd.concat([X_tr, X_v])
combined_encoded = pd.get_dummies(combined, columns=cat_cols)

X_tr_enc = combined_encoded[combined_encoded['is_train'] == 1].drop('is_train', axis=1)
X_v_enc = combined_encoded[combined_encoded['is_train'] == 0].drop('is_train', axis=1)

In [34]:
numeric_feats = X_tr_enc.dtypes[X_tr_enc.dtypes != "object"].index
skewed_feats = X_tr_enc[numeric_feats].apply(lambda x: x.skew()).sort_values(ascending=False)
high_skew = skewed_feats[abs(skewed_feats) > 0.75].index

for feat in high_skew:
    X_tr_enc[feat] = np.log1p(X_tr_enc[feat])
    X_v_enc[feat] = np.log1p(X_v_enc[feat])

### Feature Selection

In [35]:
temp_df = pd.concat([X_tr_enc, y_train], axis=1)
corr_matrix = temp_df.corr()
threshold = 0.2
relevant_cols = corr_matrix['SalePrice'][abs(corr_matrix['SalePrice']) > threshold].index.tolist()
if 'SalePrice' in relevant_cols: relevant_cols.remove('SalePrice')

X_tr_final = X_tr_enc[relevant_cols]
X_v_final = X_v_enc[relevant_cols]

In [36]:
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_final)
X_v_scaled = scaler.transform(X_v_final)

In [37]:
y_train_log = np.log1p(y_train)

In [38]:
from sklearn.linear_model import LassoCV

lasso_model = LassoCV(alphas=np.logspace(-4, -1, 30), cv=5, max_iter=10000, random_state=42)
lasso_model.fit(X_tr_scaled, y_train_log)

,"eps eps: float, default=1e-3Length of the path. ``eps=1e-3`` means that``alpha_min / alpha_max = 1e-3``.",0.001
,"n_alphas n_alphas: int, default=100Number of alphas along the regularization path... deprecated:: 1.7 `n_alphas` was deprecated in 1.7 and will be removed in 1.9. Use `alphas` instead.",'deprecated'
,"alphas alphas: array-like or int, default=NoneValues of alphas to test along the regularization path.If int, `alphas` values are generated automatically.If array-like, list of alpha values to use... versionchanged:: 1.7 `alphas` accepts an integer value which removes the need to pass `n_alphas`... deprecated:: 1.7 `alphas=None` was deprecated in 1.7 and will be removed in 1.9, at which point the default value will be set to 100.","array([0.0001..., 0.1 ])"
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: 'auto', bool or array-like of shape (n_features, n_features), default='auto'Whether to use a precomputed Gram matrix to speed upcalculations. If set to ``'auto'`` let us decide. The Grammatrix can also be passed as argument.",'auto'
,"max_iter max_iter: int, default=1000The maximum number of iterations.",10000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``.",0.0001
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"cv cv: int, cross-validation generator or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- int, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For int/None inputs, :class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: bool or int, default=FalseAmount of verbosity.",False
,"n_jobs n_jobs: int, default=NoneNumber of CPUs to use during the cross validation.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [39]:
log_preds = lasso_model.predict(X_v_scaled)
final_preds = np.expm1(log_preds)

In [40]:
rmse = np.sqrt(mean_squared_log_error(y_val, final_preds))
rmse

np.float64(0.14548037927325644)

In [41]:
mae = mean_absolute_error(y_val, final_preds)
mae

17285.18403633688

In [42]:
r2 = r2_score(y_val, final_preds)
r2

0.8971992078855165

In [43]:
used_features = np.sum(lasso_model.coef_ != 0)
used_features

np.int64(45)

In [44]:
with mlflow.start_run(run_name="Exp_4_Lasso"):
    mlflow.log_param("model_type", "LassoCV")
    mlflow.log_param("best_alpha", lasso_model.alpha_)
    mlflow.log_param("features_selected", used_features)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("stage", "Experiment 4")
    mlflow.log_param("corr_threshold", threshold)
    mlflow.log_param("final_features", len(relevant_cols))

    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )

2026/04/15 21:12:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/15 21:12:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'house_price_model' already exists. Creating a new version of this model...
2026/04/15 21:12:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: house_price_model, version 23
Created version '23' of model 'house_price_model'.


🏃 View run Exp_4_Lasso at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0/runs/2f17a23d88f34de68caef064ce5c372c
🧪 View experiment at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0


## Ex5_Outliers

### Data Cleaning

In [45]:
df_d = train.copy()
df_d = df_d.drop(df_d[(df_d['GrLivArea'] > 4000) & (df_d['SalePrice'] < 300000)].index)

In [46]:
y_d = df_d["SalePrice"]
X_d = df_d.drop("SalePrice", axis=1)
X_train_d, X_val_d, y_train_d, y_val_d = train_test_split(X_d, y_d, test_size=0.2, random_state=42)

In [47]:
num_cols = X_train_d.select_dtypes(include=[np.number]).columns
cat_cols = X_train_d.select_dtypes(include=['object']).columns

In [48]:
X_train_d[num_cols] = X_train_d[num_cols].fillna(X_train_d[num_cols].median())
X_val_d[num_cols] = X_val_d[num_cols].fillna(X_train_d[num_cols].median())

X_train_d[cat_cols] = X_train_d[cat_cols].fillna("None")
X_val_d[cat_cols] = X_val_d[cat_cols].fillna("None")

### Feature Engineering

In [49]:
for df in [X_train_d, X_val_d]:
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df.fillna(0, inplace=True)

In [50]:
X_train_d['is_train'], X_val_d['is_train'] = 1, 0
combined = pd.concat([X_train_d, X_val_d])
combined_encoded = pd.get_dummies(combined, columns=cat_cols)

X_tr_enc = combined_encoded[combined_encoded['is_train'] == 1].drop('is_train', axis=1)
X_v_enc = combined_encoded[combined_encoded['is_train'] == 0].drop('is_train', axis=1)


### Feature Selection

In [51]:
temp_df = pd.concat([X_tr_enc.reset_index(drop=True), y_train_d.reset_index(drop=True)], axis=1)

# კორელაციის გამოთვლა
corr_matrix = temp_df.corr()
relevant_cols = corr_matrix['SalePrice'][abs(corr_matrix['SalePrice']) > 0.2].index.tolist()
threshold = 0.2
if 'SalePrice' in relevant_cols:
    relevant_cols.remove('SalePrice')


In [52]:
X_tr_final = X_tr_enc[relevant_cols].fillna(0)
X_v_final = X_v_enc[relevant_cols].fillna(0)

In [53]:
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_final)
X_v_scaled = scaler.transform(X_v_final)

In [54]:
y_train_log = np.log1p(y_train_d)
lr_model_final = LinearRegression()

lr_model_final.fit(X_tr_scaled, y_train_log)
preds_e = np.expm1(lr_model_final.predict(X_v_scaled))

In [55]:
rmse = np.sqrt(mean_squared_log_error(y_val_d, preds_e))
rmse

np.float64(0.14919253724832593)

In [56]:
r2 = r2_score(y_val_d, preds_e)
r2

0.9005239904137434

In [57]:
mae = mean_absolute_error(y_val_d, preds_e)
mae

16596.712142485492

In [58]:
with mlflow.start_run(run_name="Exp_5_Outliers"):
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("stage", "Experiment 5")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    mlflow.log_param("corr_threshold", threshold)
    mlflow.log_param("final_features", len(relevant_cols))

    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )

2026/04/15 21:12:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/15 21:12:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'house_price_model' already exists. Creating a new version of this model...
2026/04/15 21:13:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: house_price_model, version 24
Created version '24' of model 'house_price_model'.


🏃 View run Exp_5_Outliers at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0/runs/ccc3da0425b34a55b61c434179046942
🧪 View experiment at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0
